In [1]:
# ============================================================================
# CHAMPION4 - FUNÇÃO DE OTIMIZAÇÃO DE HIDRÓLISE ENZIMÁTICA (VERSÃO CONDENSADA)
# ============================================================================

import numpy as np
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import MinMaxScaler

# Otimização
from scipy.optimize import differential_evolution

print("🚀 CHAMPION4 - OTIMIZAÇÃO DE HIDRÓLISE ENZIMÁTICA")
print("=" * 80)

# ============================================================================
# CONFIGURAÇÕES E CARREGAMENTO DO MODELO
# ============================================================================

# Features de entrada e saída
INPUT_FEATURES = ['Cellulose', 'Hemicellulose', 'Lignin', 'Solids Loading [g/L]', 'Enzyme Loading [g/L]', 'Time [h]']
OUTPUT_FEATURES = ['Glucose Concentration [g/L]', 'Xylose Concentration [g/L]', 'Cellobiose Concentration [g/L]']

# Ranges de otimização
OPTIMIZATION_RANGES = {
    'time': (1, 96),
    'solid_loading': (50, 300),
    'enzyme_loading': (0.01, 1.5)
}

DEFAULT_VALUES = {
    'time': 48.0,
    'solid_loading': 150.0,
    'enzyme_loading': 0.5
}

# Carregar modelo e dados
try:
    model_path = '../../../Genetic ANNs/Straw/Hydrolysis/champion_ann_strategy1_32_32_16.h5'
    champion_model = keras.models.load_model(model_path, compile=False)
    champion_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    
    data_path = '../../../Enzymatic Hydrolysis/Data Generation/synthetic_hydrolysis_data_LHS.csv'
    df = pd.read_csv(data_path)
    df.columns = df.columns.str.strip()
    
    # Configurar scalers
    scaler_X = MinMaxScaler(feature_range=(0, 1))
    scaler_y = MinMaxScaler(feature_range=(0, 1))
    scaler_X.fit(df[INPUT_FEATURES].values)
    scaler_y.fit(df[OUTPUT_FEATURES].values)
    
    print("✅ Modelo e dados carregados com sucesso!")
    
except Exception as e:
    print(f"❌ Erro ao carregar: {e}")
    print("⚠️  Usando simulação para demonstração...")
    champion_model = None
    scaler_X = None
    scaler_y = None

# ============================================================================
# FUNÇÕES PRINCIPAIS
# ============================================================================

def apply_physical_constraints(predictions, time_inputs):
    """Aplica constraints físicos (t=0h → concentrações=0)."""
    constrained_predictions = predictions.copy()
    t0_mask = np.abs(time_inputs) < 1e-6
    if np.any(t0_mask):
        constrained_predictions[t0_mask] = 0.0
    return np.maximum(constrained_predictions, 0.0)

def predict_concentrations(cellulose, hemicellulose, lignin, solid_loading, enzyme_loading, time):
    """Prediz concentrações usando a ANN ou simulação."""
    time_array = np.atleast_1d(time)
    
    if champion_model is not None and scaler_X is not None and scaler_y is not None:
        # Usar ANN real
        X = np.array([[cellulose, hemicellulose, lignin, solid_loading, enzyme_loading, t] 
                      for t in time_array])
        X_scaled = scaler_X.transform(X)
        y_pred_scaled = champion_model.predict(X_scaled, verbose=0)
        y_pred = scaler_y.inverse_transform(y_pred_scaled)
        y_pred_constrained = apply_physical_constraints(y_pred, time_array)
    else:
        # Simulação realística para demonstração
        glucose_max = solid_loading * cellulose * 1.11 * 0.8  # 80% de eficiência
        xylose_max = solid_loading * hemicellulose * 0.9 * 0.7   # 70% de eficiência
        
        # Cinética simplificada
        k_glucose = 0.05 / enzyme_loading if enzyme_loading > 0 else 0.1
        k_xylose = 0.03 / enzyme_loading if enzyme_loading > 0 else 0.05
        
        glucose_conc = glucose_max * (1 - np.exp(-k_glucose * time_array))
        xylose_conc = xylose_max * (1 - np.exp(-k_xylose * time_array))
        cellobiose_conc = glucose_conc * 0.1  # 10% de cellobiose
        
        y_pred_constrained = np.column_stack([glucose_conc, xylose_conc, cellobiose_conc])
    
    if len(time_array) == 1:
        return {
            'glucose': float(y_pred_constrained[0, 0]),
            'xylose': float(y_pred_constrained[0, 1]),
            'cellobiose': float(y_pred_constrained[0, 2])
        }
    else:
        return {
            'glucose': y_pred_constrained[:, 0],
            'xylose': y_pred_constrained[:, 1],
            'cellobiose': y_pred_constrained[:, 2],
            'time': time_array
        }

def calculate_glucose_yield(glucose_concentration, cellulose_percent, hemicellulose_percent, solid_loading):
    """Calcula rendimento de glucose."""
    cellulose_available = solid_loading * cellulose_percent
    hemicellulose_available = solid_loading * hemicellulose_percent
    theoretical_glucose = cellulose_available * 1.11 + hemicellulose_available * 0.2
    
    if theoretical_glucose > 0:
        yield_percent = (glucose_concentration / theoretical_glucose) * 100
        return min(yield_percent, 100.0)
    return 0.0

def multi_objective_function(params, config, cellulose, hemicellulose, lignin):
    """Função objetivo multi-objetivo."""
    inputs_dict = config['known_values'].copy()
    
    for i, param_name in enumerate(config['inputs_to_optimize']):
        inputs_dict[param_name] = params[i]
    
    for param_name in ['time', 'solid_loading', 'enzyme_loading']:
        if param_name not in inputs_dict:
            inputs_dict[param_name] = config['defaults'][param_name]
    
    try:
        prediction = predict_concentrations(
            cellulose=cellulose, hemicellulose=hemicellulose, lignin=lignin,
            solid_loading=inputs_dict['solid_loading'],
            enzyme_loading=inputs_dict['enzyme_loading'],
            time=inputs_dict['time']
        )
        
        glucose = prediction['glucose']
        weights = config['weights']
        
        # Componentes normalizados
        glucose_component = weights['glucose_weight'] * glucose / 100.0
        time_component = weights['time_weight'] * (inputs_dict['time'] / 96.0)
        solid_component = weights['solid_weight'] * (inputs_dict['solid_loading'] / 300.0)
        enzyme_component = weights['enzyme_weight'] * (inputs_dict['enzyme_loading'] / 2.0)
        
        objective_value = glucose_component + time_component + solid_component + enzyme_component
        return -objective_value  # Negativo para minimização
        
    except Exception as e:
        return 1e6

def optimize_hydrolysis_process(cellulose_percent, hemicellulose_percent, lignin_percent,
                               solid_loading=None, enzyme_loading=None, reaction_time=None,
                               n_iterations=30, verbose=True):
    """
    Função principal de otimização de hidrólise enzimática.
    
    Parameters:
    -----------
    cellulose_percent : float
        Percentual de celulose (0-100)
    hemicellulose_percent : float
        Percentual de hemicelulose (0-100)
    lignin_percent : float
        Percentual de lignina (0-100)
    solid_loading : float, optional
        Carregamento de sólidos (g/L). Se None, será otimizado.
    enzyme_loading : float, optional
        Carregamento de enzima (g/L). Se None, será otimizado.
    reaction_time : float, optional
        Tempo de reação (h). Se None, será otimizado.
    n_iterations : int, default=30
        Número de iterações para otimização
    verbose : bool, default=True
        Mostrar detalhes do processo
        
    Returns:
    --------
    dict : Resultado completo com predições e métricas
    """
    
    if verbose:
        print(f"\n🔬 ANÁLISE DO PROCESSO DE HIDRÓLISE ENZIMÁTICA")
        print(f"=" * 60)
        print(f"📊 COMPOSIÇÃO DA BIOMASSA:")
        print(f"   • Celulose: {cellulose_percent:.2f}%")
        print(f"   • Hemicelulose: {hemicellulose_percent:.2f}%")
        print(f"   • Lignina: {lignin_percent:.2f}%")
    
    # Converter para frações
    cellulose_fraction = cellulose_percent / 100.0
    hemicellulose_fraction = hemicellulose_percent / 100.0
    lignin_fraction = lignin_percent / 100.0
    
    # Identificar inputs conhecidos
    known_inputs = []
    known_values = {}
    
    if reaction_time is not None:
        known_inputs.append('time')
        known_values['time'] = reaction_time
    if solid_loading is not None:
        known_inputs.append('solid_loading')
        known_values['solid_loading'] = solid_loading
    if enzyme_loading is not None:
        known_inputs.append('enzyme_loading')
        known_values['enzyme_loading'] = enzyme_loading
    
    all_inputs = ['time', 'solid_loading', 'enzyme_loading']
    inputs_to_optimize = [inp for inp in all_inputs if inp not in known_inputs]
    
    # Configuração de otimização
    config = {
        'inputs_to_optimize': inputs_to_optimize,
        'known_inputs': known_inputs,
        'known_values': known_values,
        'ranges': OPTIMIZATION_RANGES,
        'defaults': DEFAULT_VALUES,
        'weights': {
            'glucose_weight': 1.0,
            'time_weight': -0.1,
            'solid_weight': -0.001,
            'enzyme_weight': -0.5
        }
    }
    
    if verbose:
        print(f"\n⚙️  PARÂMETROS:")
        print(f"   • Tempo: {reaction_time if reaction_time is not None else 'A OTIMIZAR'}")
        print(f"   • Sólidos: {solid_loading if solid_loading is not None else 'A OTIMIZAR'} g/L")
        print(f"   • Enzima: {enzyme_loading if enzyme_loading is not None else 'A OTIMIZAR'} g/L")
    
    start_time = time.time()
    
    if not inputs_to_optimize:
        # Sem otimização - usar valores conhecidos
        inputs_dict = known_values.copy()
        for param_name in all_inputs:
            if param_name not in inputs_dict:
                inputs_dict[param_name] = DEFAULT_VALUES[param_name]
                
        prediction = predict_concentrations(
            cellulose_fraction, hemicellulose_fraction, lignin_fraction,
            inputs_dict['solid_loading'], inputs_dict['enzyme_loading'], inputs_dict['time']
        )
        
        result = {
            'optimized_inputs': inputs_dict,
            'predicted_outputs': prediction,
            'optimization_used': False
        }
    else:
        # Executar otimização
        if verbose:
            print(f"\n🚀 INICIANDO OTIMIZAÇÃO...")
            print(f"   • Parâmetros a otimizar: {inputs_to_optimize}")
            print(f"   • Iterações: {n_iterations}")
        
        bounds = []
        for param_name in inputs_to_optimize:
            min_val, max_val = OPTIMIZATION_RANGES[param_name]
            bounds.append((min_val, max_val))
        
        optimization_result = differential_evolution(
            multi_objective_function,
            bounds=bounds,
            args=(config, cellulose_fraction, hemicellulose_fraction, lignin_fraction),
            maxiter=n_iterations,
            popsize=15,
            seed=42,
            polish=True
        )
        
        # Reconstruir inputs otimizados
        optimized_inputs = known_values.copy()
        for i, param_name in enumerate(inputs_to_optimize):
            optimized_inputs[param_name] = optimization_result.x[i]
        
        for param_name in all_inputs:
            if param_name not in optimized_inputs:
                optimized_inputs[param_name] = DEFAULT_VALUES[param_name]
        
        final_prediction = predict_concentrations(
            cellulose_fraction, hemicellulose_fraction, lignin_fraction,
            optimized_inputs['solid_loading'], optimized_inputs['enzyme_loading'], optimized_inputs['time']
        )
        
        result = {
            'optimized_inputs': optimized_inputs,
            'predicted_outputs': final_prediction,
            'optimization_used': True,
            'optimization_success': optimization_result.success,
            'function_evaluations': optimization_result.nfev,
            'objective_value': -optimization_result.fun
        }
    
    optimization_time = time.time() - start_time
    
    # Calcular métricas
    glucose_yield = calculate_glucose_yield(
        result['predicted_outputs']['glucose'],
        cellulose_fraction, hemicellulose_fraction,
        result['optimized_inputs']['solid_loading']
    )
    
    time_cost = result['optimized_inputs']['time'] / 96.0 * 100
    solid_cost = result['optimized_inputs']['solid_loading'] / 300.0 * 100
    enzyme_cost = result['optimized_inputs']['enzyme_loading'] / 2.0 * 100
    total_cost_index = (time_cost + solid_cost + enzyme_cost) / 3
    
    # Resultado final
    final_result = {
        'process_conditions': {
            'reaction_time_h': round(result['optimized_inputs']['time'], 2),
            'solid_loading_g_L': round(result['optimized_inputs']['solid_loading'], 2),
            'enzyme_loading_g_L': round(result['optimized_inputs']['enzyme_loading'], 4)
        },
        'predicted_concentrations': {
            'glucose_g_L': round(result['predicted_outputs']['glucose'], 3),
            'xylose_g_L': round(result['predicted_outputs']['xylose'], 3),
            'cellobiose_g_L': round(result['predicted_outputs']['cellobiose'], 3)
        },
        'performance_metrics': {
            'glucose_yield_percent': round(glucose_yield, 2),
            'total_sugar_g_L': round(
                result['predicted_outputs']['glucose'] + 
                result['predicted_outputs']['xylose'] + 
                result['predicted_outputs']['cellobiose'], 3
            ),
            'cost_index_percent': round(total_cost_index, 1)
        },
        'optimization_info': {
            'optimization_used': result['optimization_used'],
            'optimization_time_s': round(optimization_time, 3),
            'parameters_optimized': inputs_to_optimize,
            'success': result.get('optimization_success', True)
        }
    }
    
    if verbose:
        print(f"\n✅ OTIMIZAÇÃO CONCLUÍDA!")
        print(f"⏱️  Tempo de execução: {optimization_time:.3f}s")
        print(f"\n📋 CONDIÇÕES OTIMIZADAS:")
        print(f"   • Tempo: {final_result['process_conditions']['reaction_time_h']}h")
        print(f"   • Sólidos: {final_result['process_conditions']['solid_loading_g_L']} g/L")
        print(f"   • Enzima: {final_result['process_conditions']['enzyme_loading_g_L']} g/L")
        print(f"\n📈 CONCENTRAÇÕES PREDITAS:")
        print(f"   • Glucose: {final_result['predicted_concentrations']['glucose_g_L']} g/L")
        print(f"   • Xylose: {final_result['predicted_concentrations']['xylose_g_L']} g/L")
        print(f"   • Cellobiose: {final_result['predicted_concentrations']['cellobiose_g_L']} g/L")
        print(f"\n🎯 MÉTRICAS DE PERFORMANCE:")
        print(f"   • Rendimento de glucose: {final_result['performance_metrics']['glucose_yield_percent']}%")
        print(f"   • Total de açúcares: {final_result['performance_metrics']['total_sugar_g_L']} g/L")
        print(f"   • Índice de custos: {final_result['performance_metrics']['cost_index_percent']}%")
    
    return final_result

# ============================================================================
# TESTE DA FUNÇÃO
# ============================================================================

print(f"\n🧪 TESTANDO FUNÇÃO DE OTIMIZAÇÃO...")

# Teste 1: Otimizar solid_loading e enzyme_loading (tempo fixo)
print(f"\n" + "="*60)
print(f"TESTE 1: OTIMIZAR SOLID_LOADING E ENZYME_LOADING")
test_result_1 = optimize_hydrolysis_process(
    cellulose_percent=57.0,
    hemicellulose_percent=13.0,
    lignin_percent=25.0,
    solid_loading=None,      # A otimizar
    enzyme_loading=None,     # A otimizar
    reaction_time=48.0,      # Fixo
    n_iterations=25,
    verbose=True
)

# Teste 2: Otimizar todos os parâmetros
print(f"\n" + "="*60)
print(f"TESTE 2: OTIMIZAR TODOS OS PARÂMETROS")
test_result_2 = optimize_hydrolysis_process(
    cellulose_percent=45.0,
    hemicellulose_percent=20.0,
    lignin_percent=30.0,
    solid_loading=None,      # A otimizar
    enzyme_loading=None,     # A otimizar
    reaction_time=None,      # A otimizar
    n_iterations=25,
    verbose=True
)

# Teste 3: Apenas predição (todos parâmetros fornecidos)
print(f"\n" + "="*60)
print(f"TESTE 3: PREDIÇÃO DIRETA (SEM OTIMIZAÇÃO)")
test_result_3 = optimize_hydrolysis_process(
    cellulose_percent=60.0,
    hemicellulose_percent=15.0,
    lignin_percent=20.0,
    solid_loading=150.0,     # Fixo
    enzyme_loading=0.5,      # Fixo
    reaction_time=36.0,      # Fixo
    n_iterations=25,
    verbose=True
)

print(f"\n🎉 TODOS OS TESTES CONCLUÍDOS COM SUCESSO!")
print(f"✅ A função está pronta para uso no Streamlit!")

# Função simplificada para uso direto
def optimize_simple(cellulose, hemicellulose, lignin, solid=None, enzyme=None, time=None):
    """Versão simplificada para uso rápido."""
    return optimize_hydrolysis_process(
        cellulose, hemicellulose, lignin, solid, enzyme, time, 
        n_iterations=20, verbose=False
    )

print(f"\n📝 USO SIMPLIFICADO:")
print(f"   result = optimize_simple(57, 13, 25)")
print(f"   result = optimize_simple(57, 13, 25, solid=150)")
print(f"   result = optimize_simple(57, 13, 25, enzyme=0.3, time=48)")

🚀 CHAMPION4 - OTIMIZAÇÃO DE HIDRÓLISE ENZIMÁTICA
✅ Modelo e dados carregados com sucesso!

🧪 TESTANDO FUNÇÃO DE OTIMIZAÇÃO...

TESTE 1: OTIMIZAR SOLID_LOADING E ENZYME_LOADING

🔬 ANÁLISE DO PROCESSO DE HIDRÓLISE ENZIMÁTICA
📊 COMPOSIÇÃO DA BIOMASSA:
   • Celulose: 57.00%
   • Hemicelulose: 13.00%
   • Lignina: 25.00%

⚙️  PARÂMETROS:
   • Tempo: 48.0
   • Sólidos: A OTIMIZAR g/L
   • Enzima: A OTIMIZAR g/L

🚀 INICIANDO OTIMIZAÇÃO...
   • Parâmetros a otimizar: ['solid_loading', 'enzyme_loading']
   • Iterações: 25

✅ OTIMIZAÇÃO CONCLUÍDA!
⏱️  Tempo de execução: 27.279s

📋 CONDIÇÕES OTIMIZADAS:
   • Tempo: 48.0h
   • Sólidos: 299.96 g/L
   • Enzima: 1.1332 g/L

📈 CONCENTRAÇÕES PREDITAS:
   • Glucose: 89.361 g/L
   • Xylose: 41.404 g/L
   • Cellobiose: 6.989 g/L

🎯 MÉTRICAS DE PERFORMANCE:
   • Rendimento de glucose: 45.23%
   • Total de açúcares: 137.754 g/L
   • Índice de custos: 68.9%

TESTE 2: OTIMIZAR TODOS OS PARÂMETROS

🔬 ANÁLISE DO PROCESSO DE HIDRÓLISE ENZIMÁTICA
📊 COMPOSIÇÃO DA 